# Slide 29 demo: Amazon Comprehend entity detection and custom classification

**Class scenario:** The RAG ingestion pipeline receives policy and technical documents. Before indexing, extract entities to enrich metadata. For a business-specific category such as `SECURITY_POLICY` or `TROUBLESHOOTING`, use a **trained custom classifier**.

Run top to bottom in VS Code. Cells 1–3 preview the flow without AWS and are labeled illustrative. To call the real service, set `RUN_AWS = True` in cell 1 and provide credentials with `comprehend:DetectEntities`. Custom classification additionally requires a trained model, an active endpoint, and `comprehend:ClassifyDocument`. Never put real customer documents into a classroom demo.


## 1. Configuration and sample documents


In [ ]:
RUN_AWS = False  # Change to True to call Amazon Comprehend
AWS_REGION = "<AWS_REGION>"  # e.g. "us-east-1"
LANGUAGE_CODE = "en"
CLASSIFIER_ENDPOINT_ARN = ""  # Optional: active document-classifier-endpoint ARN
MIN_ENTITY_SCORE = 0.80
MIN_CLASS_SCORE = 0.70

DOCUMENTS = [
    {"id":"doc-001", "text":"Amazon S3 access policies for Acme Corp were reviewed in Mumbai on 18 September 2026. Contact the security team about Policy SEC-204."},
    {"id":"doc-002", "text":"A customer received AccessDeniedException when uploading to Amazon S3. Check IAM permissions and the bucket policy."},
]
for d in DOCUMENTS:
    print(d["id"], d["text"])


## 2. Detect named entities

With `RUN_AWS=True`, this executes the real `DetectEntities` API. The API returns text, type, confidence score, and offsets. The preview rows when `RUN_AWS=False` are **illustrative**, not predicted Comprehend results. A service may label an organization or place differently than you expect.


In [ ]:
if RUN_AWS:
    import boto3
    comprehend = boto3.client("comprehend", region_name=AWS_REGION)
    def detect_entities(text):
        return comprehend.detect_entities(Text=text, LanguageCode=LANGUAGE_CODE)["Entities"]
else:
    SAMPLE_ENTITIES = {
        "doc-001": [
            {"Text":"Amazon S3", "Type":"ORGANIZATION", "Score":0.96},
            {"Text":"Acme Corp", "Type":"ORGANIZATION", "Score":0.93},
            {"Text":"Mumbai", "Type":"LOCATION", "Score":0.98},
            {"Text":"18 September 2026", "Type":"DATE", "Score":0.97},
        ],
        "doc-002": [{"Text":"Amazon S3", "Type":"ORGANIZATION", "Score":0.96}],
    }
    def detect_entities(text, document_id):
        return SAMPLE_ENTITIES[document_id]

ALL_ENTITIES = {}
for doc in DOCUMENTS:
    entities = detect_entities(doc["text"]) if RUN_AWS else detect_entities(doc["text"],doc["id"])
    ALL_ENTITIES[doc["id"]] = entities
    print(f"\n{doc['id']}:")
    for e in entities:
        print(f"  {e['Text']:<24} {e['Type']:<16} confidence={e['Score']:.2f}")


## 3. Turn reviewed output into metadata

A high score alone does not establish a document's owner, classification, or access policy. The stable source ID, access label, and policy ID below come from **trusted source metadata or explicit validation**, not from Comprehend's inferred entities. Review and normalize entity labels before using them as filters.


In [ ]:
SOURCE_METADATA = {
    "doc-001": {"source_id":"s3://demo-docs/policy-SEC-204.txt", "access":"staff", "policy_id":"SEC-204"},
    "doc-002": {"source_id":"s3://demo-docs/s3-error-guide.txt", "access":"staff", "policy_id":None},
}
ALLOWED_ENTITY_TYPES = {"ORGANIZATION", "LOCATION", "DATE"}
def reviewed_metadata(doc_id):
    trusted = dict(SOURCE_METADATA[doc_id])
    trusted["candidate_entities"] = [
        {"value":e["Text"], "type":e["Type"], "score":round(e["Score"],3)}
        for e in ALL_ENTITIES[doc_id]
        if e["Score"] >= MIN_ENTITY_SCORE and e["Type"] in ALLOWED_ENTITY_TYPES
    ]
    return trusted

for doc in DOCUMENTS:
    print(doc["id"], reviewed_metadata(doc["id"]))


## 4. Optional: custom classification

`DetectEntities` uses pretrained entity categories. To predict **your own document classes**, prepare labeled examples, train an Amazon Comprehend custom classifier, evaluate it, create an inference endpoint, and pass the endpoint ARN to `ClassifyDocument`. An endpoint can incur ongoing charges while it is active; delete it after the demo if it was created only for class.

For a console setup: **Amazon Comprehend → Customization → Custom classification → Create new model**; supply training data in S3 and wait for training. Review the resulting evaluation metrics. Then create an endpoint for that model in the same Region. This notebook does not train or create billable resources.


In [ ]:
if not RUN_AWS or not CLASSIFIER_ENDPOINT_ARN:
    print("Custom classification skipped. Set RUN_AWS=True and CLASSIFIER_ENDPOINT_ARN to an active endpoint to call ClassifyDocument.")
else:
    for doc in DOCUMENTS:
        result = comprehend.classify_document(
            Text=doc["text"], EndpointArn=CLASSIFIER_ENDPOINT_ARN
        )
        print(f"\n{doc['id']}:")
        for item in result.get("Classes", []):
            print(item["Name"], round(item["Score"],3),
                  "ACCEPT" if item["Score"] >= MIN_CLASS_SCORE else "REVIEW")
        for item in result.get("Labels", []):
            print("Multi-label:",item["Name"],round(item["Score"],3))


## 5. Instructor prompts and exam check

1. Ask: *Which entities did Comprehend find, and which expected identifiers were absent?* Notice that a policy ID or exception name is not guaranteed to be a supported named entity.
2. Ask: *Can `DetectEntities` decide whether a document is `SECURITY_POLICY`?* No: that is a business label and needs a custom classifier or an explicit rule.
3. Ask: *Can `DetectEntities` parse a scanned PDF or enforce a user's access rights?* No: plan OCR/parsing where needed, and enforce authorization separately.
4. Ask: *What happens to low-confidence results?* Review or quarantine them; do not silently turn guesses into authoritative metadata.
5. If you demonstrate custom classification, compare the predicted class with a human-labeled example and explain the confidence threshold.

**Exam-style distinction:** Named entities → `DetectEntities`; your own document categories → train a custom classifier and call `ClassifyDocument` for real-time inference. For an S3 batch corpus, consider an asynchronous classification job instead of maintaining a real-time endpoint.

AWS documentation: [DetectEntities](https://docs.aws.amazon.com/comprehend/latest/APIReference/API_DetectEntities.html), [Custom classification](https://docs.aws.amazon.com/comprehend/latest/dg/how-document-classification.html), [Real-time inference and endpoints](https://docs.aws.amazon.com/comprehend/latest/dg/class-sync-api.html).
